# El puesto de observación: hablarle al robot desde una notebook

## Manual de encendido (en tu terminal, paso a paso — leer antes de ejecutar nada)

El kernel de esta notebook debe correr **en la VM** (ahí viven `rclpy` y los buzones
del robot). Ningún Python de tu laptop sirve. Todo el arranque se hace en tu
terminal; esta notebook recién entra en escena en el paso 5.

> **Atajo**: `bash scripts/lab_up.sh` hace los pasos 1-4 de un saque (y esa
> terminal queda siendo el túnel). Los pasos de abajo son lo mismo, a mano.

**1. Prender la VM y averiguar su IP** — la IP **cambia en cada encendido**, por
eso no está escrita en ningún lado y se le pregunta a Azure:
```
az vm start -g rg-go2-lab -n vm-go2-isaac
az vm show  -g rg-go2-lab -n vm-go2-isaac -d --query publicIps -o tsv
```

**2. Lanzar el simulador** (si no está corriendo) — reemplazá `<IP>` por la del paso 1:
```
ssh lucas@<IP>
~/go2-lab/launch_port.sh        # tarda ~2 min; podés seguir con los pasos
```

**3. Lanzar Jupyter en la VM** (si no está corriendo) — en esa misma terminal ssh:
```
source /opt/ros/jazzy/setup.bash && setsid nohup ~/venvs/nb/bin/jupyter lab \
  --no-browser --port 8888 --ip 127.0.0.1 --ServerApp.token=go2lab \
  --notebook-dir ~/go2-lab/notebooks > ~/jupyter.log 2>&1 &
```

**4. El túnel** — en **otra** terminal tuya (y dejarla abierta: es el cable):
```
ssh -L 8888:localhost:8888 lucas@<IP>
```

**5. Conectar el kernel en VSCode**: abrir esta notebook → **Select Kernel** →
**Select Another Kernel** → **Existing Jupyter Server** → pegar:
```
http://localhost:8888/?token=go2lab
```
(Alternativa sin VSCode: abrir `http://localhost:8888/lab?token=go2lab` en el
navegador y usar la copia que vive en la VM.)

**6. El cliente de Isaac** conectado a la IP del paso 1, en la otra mitad de la
pantalla — para ver al robot reaccionar a cada celda.

**Al terminar el día**: `bash scripts/lab_down.sh` (cierra procesos y apaga la
VM — deja de facturar).

---

## Cómo funciona esto

El kernel corre **en la VM** — la misma máquina donde vive el simulador. Eso
significa que esta notebook está *en la misma calle* que el robot: puede escuchar
sus buzones y escribirle cartas, directo, con `rclpy` (la librería Python de ROS 2).
Tu laptop es solo la ventana (el archivo se edita acá y se versiona en git; el
código se ejecuta allá).

Ejecutá las celdas en orden con `Shift+Enter`.

## Celda 1 — Entrar a la calle

`rclpy.init()` arranca la maquinaria DDS de ESTE proceso (la notebook), y crear un
`node` nos da identidad en la red: a partir de acá, **la notebook es un vecino más** —
igual que el simulador, igual que lo será el robot real. No nos "conectamos al robot":
nos paramos en su misma calle.

In [ ]:
import time
import rclpy
from geometry_msgs.msg import Twist
from nav_msgs.msg import Odometry

if not rclpy.ok():          # guarda: permite re-ejecutar la celda sin error
    rclpy.init()
node = rclpy.create_node("observatorio")
print("Somos un vecino de la calle ROS 2. Nombre del nodo:", node.get_name())

## Celda 2 — El censo de buzones

Le preguntamos a la calle qué topics existen. Cada uno tiene **nombre** y **tipo de
mensaje** (el contrato de qué datos lleva). Los `/unitree_go2/...` los creó el bridge
del simulador.

In [ ]:
time.sleep(1.0)  # darle un segundo al descubrimiento DDS
for name, types in sorted(node.get_topic_names_and_types()):
    print(f"{name:55s} {types[0]}")

## Celda 3 — Qué **devuelve** el robot: un mensaje de odometría, entero

Nos suscribimos a `/unitree_go2/odom` y capturamos un mensaje para mirarlo por dentro.
Esto es lo que el robot cuenta de sí mismo ~50 veces por segundo:

- `pose.position` — dónde está (x, y, z en metros; z≈0.42 = de pie)
- `pose.orientation` — hacia dónde mira (quaternion)
- `twist.linear / angular` — a qué velocidad se mueve y rota ahora mismo
- `header.stamp` — cuándo se midió

El callback es el patrón de siempre: "cuando llegue carta, guardala" — idéntico a tu
`00_read_lowstate.py` de la era MuJoCo, con formato estándar en vez de propietario.

In [ ]:
latest = {"odom": None, "count": 0}

def on_odom(msg):
    latest["odom"] = msg
    latest["count"] += 1

sub = node.create_subscription(Odometry, "/unitree_go2/odom", on_odom, 10)

# atender el buzon hasta que caiga el primer mensaje
t0 = time.time()
while latest["odom"] is None and time.time() - t0 < 10:
    rclpy.spin_once(node, timeout_sec=0.2)

m = latest["odom"]
if m is None:
    print("No llego nada: esta corriendo el simulador?")
else:
    p, o = m.pose.pose.position, m.pose.pose.orientation
    v, w = m.twist.twist.linear, m.twist.twist.angular
    print(f"posicion      x={p.x:+.3f}  y={p.y:+.3f}  z={p.z:+.3f}   <- z=0.42 es DE PIE")
    print(f"orientacion   quaternion ({o.w:+.3f}, {o.x:+.3f}, {o.y:+.3f}, {o.z:+.3f})")
    print(f"vel lineal    x={v.x:+.3f}  y={v.y:+.3f}  z={v.z:+.3f}  [m/s]")
    print(f"vel angular   z={w.z:+.3f}  [rad/s]")
    print(f"mensajes recibidos mientras esperabamos: {latest['count']}")

## Celda 4 — Qué se le **pide**: el mensaje Twist (y verlo obedecer)

El contrato de entrada es un mensaje `Twist`: en la práctica, para este robot,
**tres números** — `linear.x` (adelante), `linear.y` (costado), `angular.z` (giro).
Nada de distancias, nada de destinos: solo velocidades sostenidas.

**Mirá el cliente de Isaac al ejecutar**: esta celda publica "adelante a 0.5 m/s"
durante 3 segundos (10 cartas por segundo). El robot va a trotar mientras la celda
corre... y **frenarse solo** medio segundo después de terminar — ese frenado
automático es el *watchdog* que le agregamos al bridge: sin carta fresca en 0.5 s,
comando a cero. Un hombre muerto de seguridad, en acción.

In [ ]:
pub = node.create_publisher(Twist, "/unitree_go2/cmd_vel", 10)

cmd = Twist()
cmd.linear.x = 0.5      # adelante, medio metro por segundo
print("La carta que vamos a mandar (10 veces por segundo, 3 segundos):")
print(f"  linear  = ({cmd.linear.x}, {cmd.linear.y}, {cmd.linear.z})")
print(f"  angular = ({cmd.angular.x}, {cmd.angular.y}, {cmd.angular.z})")

t0 = time.time()
while time.time() - t0 < 3.0:
    pub.publish(cmd)
    rclpy.spin_once(node, timeout_sec=0.0)
    time.sleep(0.1)
print("\nListo. Mira el cliente: el robot deberia frenarse solo (watchdog).")

## Celda 5 — El lazo completo, en vivo: ordenar Y escuchar a la vez

Ahora las dos cosas juntas: publicamos velocidad mientras leemos la odometría —
**el ida y vuelta completo**, numérico acá, visual en el cliente. Fijate cómo la
velocidad reportada (`vel`) persigue a la comandada (0.5) sin alcanzarla exacto:
la policy no es perfecta, es *suficiente*.

In [ ]:
cmd = Twist(); cmd.linear.x = 0.5
t0 = time.time()
print(f"{'t':>5} {'x':>8} {'y':>8} {'vel x':>7}")
while time.time() - t0 < 6.0:
    pub.publish(cmd)
    rclpy.spin_once(node, timeout_sec=0.0)
    m = latest["odom"]
    if m and int((time.time() - t0) * 10) % 5 == 0:   # imprimir ~cada 0.5 s
        p, v = m.pose.pose.position, m.twist.twist.linear
        print(f"{time.time()-t0:5.1f} {p.x:8.3f} {p.y:8.3f} {v.x:7.3f}")
    time.sleep(0.1)
print("fin — el watchdog lo frena en ~0.5 s")

## Celda 6 — "Caminá 3 metros" (tu primer navegador)

Pediste entender qué pasa cuando ordenás "caminá 5 metros". La clave: **esa orden
no existe en este piso** — `cmd_vel` solo entiende velocidades. "Caminar N metros"
hay que *construirlo* un piso más arriba: un loop que mide cuánto avanzó (odometría)
y deja de empujar al llegar. Eso es — ni más ni menos — **un navegador**: el asiento
de arriba de la pirámide, en 15 líneas.

```
  "3 metros"  →  [ESTA CELDA: mide y decide]  →  cmd_vel  →  policy  →  motores
```

En el cliente: va a caminar ~3 metros y frenar solo — esta vez no por watchdog,
sino porque *la celda decidió que llegó*.

In [ ]:
import math

META_METROS = 3.0

# 1. donde estoy ahora? (leer odometria fresca)
rclpy.spin_once(node, timeout_sec=0.5)
p0 = latest["odom"].pose.pose.position
x0, y0 = p0.x, p0.y
print(f"Partida: ({x0:.2f}, {y0:.2f}). Objetivo: avanzar {META_METROS} m.")

cmd = Twist(); cmd.linear.x = 0.5
recorrido = 0.0
while recorrido < META_METROS:
    pub.publish(cmd)                          # empujar
    rclpy.spin_once(node, timeout_sec=0.0)    # atender el buzon
    p = latest["odom"].pose.pose.position
    recorrido = math.hypot(p.x - x0, p.y - y0) # medir cuanto avance
    time.sleep(0.1)

# llegue: frenar explicitamente (sin esperar al watchdog)
pub.publish(Twist())
print(f"LLEGUE: recorri {recorrido:.2f} m. Posicion final: ({p.x:.2f}, {p.y:.2f})")

## Qué acabás de hacer

En seis celdas recorriste la arquitectura entera, con las manos:

1. Te paraste en la calle del robot (rclpy = vecino nuevo).
2. Censaste los buzones.
3. Leíste el **contrato de salida** (Odometry: posición, orientación, velocidades).
4. Escribiste el **contrato de entrada** (Twist: 3 números) y lo viste obedecer.
5. Cerraste el lazo: ordenar y medir a la vez.
6. Construiste el piso que faltaba: **un navegador** que convierte "N metros"
   (concepto que el robot no entiende) en velocidades (que sí entiende).

Y de regalo viste trabajar al watchdog dos veces. Todo lo que hagamos de acá en
adelante — skills, agente, misiones — son versiones más elaboradas de la celda 6:
programas que miden, deciden y escriben tres números en un buzón.